In [1]:
"""
Sinh response XSTest (no defense) bằng llama-3.1-8b-instant qua Groq.
Chỉ giữ safe prompts.
Output: data gốc (id, prompt, type, label) + cột response.

pip install openai pandas tqdm
"""
import os, time
import pandas as pd
from openai import OpenAI, RateLimitError, APIError
from tqdm import tqdm

from kaggle_secrets import UserSecretsClient
os.environ["GROQ_API_KEY"] = UserSecretsClient().get_secret("GROQ_API_KEY")

# --- CONFIG ---
XSTEST_PATH  = "/kaggle/input/datasets/honglchnguyn87/xstestdata/data/xstest_safeprompts.csv"   # file XSTest gốc (có cột prompt, type, label...)
OUTPUT_PATH  = "/kaggle/working/xstest_llama31_8b_responses_250prompt.csv"

TARGET_MODEL     = "llama-3.1-8b-instant"   # target chạy API (theo quy ước)
MAX_TOKENS       = 512
REQUEST_INTERVAL = 2.0
CHECKPOINT_EVERY = 25

client = OpenAI(api_key=os.environ["GROQ_API_KEY"], base_url="https://api.groq.com/openai/v1")

# --- Load + lọc chỉ safe ---
df = pd.read_csv(XSTEST_PATH)
df["label"] = df["label"].str.strip().str.lower()
df = df[df["label"] == "safe"].reset_index(drop=True)
print(f"Giữ {len(df)} safe prompts (đã bỏ unsafe)")

# --- Gọi target (no defense = gọi thẳng prompt) ---
def generate(prompt):
    delay = 5
    for _ in range(6):
        try:
            resp = client.chat.completions.create(
                model=TARGET_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
                max_completion_tokens=MAX_TOKENS,
            )
            return resp.choices[0].message.content or ""
        except RateLimitError:
            print(f"Rate limited, chờ {delay}s..."); time.sleep(delay); delay = min(delay*2, 60)
        except APIError as e:
            print(f"API error: {e}, chờ {delay}s..."); time.sleep(delay); delay = min(delay*2, 60)
    return ""

# --- Chạy, có checkpoint ---
responses = []
for i, row in enumerate(tqdm(df.itertuples(), total=len(df))):
    time.sleep(REQUEST_INTERVAL)
    r = generate(row.prompt)
    responses.append(r)
    tqdm.write(f"[{row.id}] len={len(r)} | {str(r)[:50]}")

    if (i + 1) % CHECKPOINT_EVERY == 0:
        df.iloc[:len(responses)].assign(response=responses).to_csv(OUTPUT_PATH, index=False)

df["response"] = responses
df.to_csv(OUTPUT_PATH, index=False)
print(f"\nSaved -> {OUTPUT_PATH}")
print(f"Response rỗng: {(df['response'].str.strip() == '').sum()}/{len(df)}")

KeyboardInterrupt: 